# Monarch - Gate 2: 400-item corpus NAA scan (Kaggle GPU)

Runs the full text->fMRI cascade (gTTS -> WhisperX -> Llama-3.2-3B + Wav2Vec-BERT -> TRIBE v2)
over the four-category corpus and writes one NAA row per item. This is the only step in the
thesis that needs a GPU and the only one that cannot be repeated cheaply.

**Cost, measured rather than assumed.** The observed rate on a Tesla P100 is 187.5 s/item,
so 400 items is 20.8 GPU-hours. A T4 is projected at about 110 s/item (12.2 h), but no run
has yet been allocated one, so that figure is a projection and not a measurement. The
earlier 75 s/item in this header was never observed and has been removed.

**Quota.** 30 GPU-h per week, refreshing weekly. Read it from the typed fields of
`quota_view()`, not from the response's JSON repr: the repr prints
`totalTimeAllowed "21600s"` while `total_time_allowed` is `1 day, 6:00:00` (108000 s), and
believing the repr turns a 30 h allowance into an imaginary 6 h one.

**Sessions, not quota, are the binding constraint.** A Kaggle session caps at 12 h and 400
items need 20.8 h on a P100, so the corpus takes at least two sessions whatever the quota
says. Each run is capped by `--limit` in the scan cell so it ends on its own terms and its
output is published; a session killed by any external limit is not known to publish output,
and the partial CSV is the entire point of the run.

### Before you press Run All

1. **Settings -> Accelerator = GPU.** T4 x2 is worth selecting, but the scheduler ignores it
   when two T4s are not free: five consecutive sessions drew a P100, including one pushed
   through the API with `machine_shape=gpuT4x2`. The notebook now accepts a P100 and says so.
2. **Add-ons -> Secrets -> `HF_TOKEN`**, from an account that has accepted the
   Llama-3.2 licence. Without it the embedding stage 401s.
3. **Add data -> your corpus dataset**, containing `corpus.csv` (400 rows, built by
   `scripts/build_corpus.py`). Any dataset name works; the notebook globs for the file.
4. The branch below must be **pushed to GitHub**. Kaggle clones the repo; anything sitting
   uncommitted on the laptop does not exist here.
5. **Resuming a killed run:** upload the partial `corpus_naa.csv` as a dataset too. The
   corpus cell seeds `/kaggle/working` from it and `batch_naa.py` skips every item already
   scanned.

### Rules this notebook does not break

- No `alpha_hat` is quoted. Calibration is a separate, free CPU step run after the scan.
- Undefined NAA rows are written empty and counted, never dropped or filled.
- The summary at the end reports counts only. Analysis happens in `analyze_corpus.py`.

In [ ]:
%%bash
# tribev2 declares torch>=2.5.1,<2.7 and Kaggle ships 2.10, so the image is installed against
# a version the model was never built for. Rather than branch per card, put the declared
# version in place first: cu121 wheels target sm_50 through sm_90, which covers both the T4
# (sm_75) and the P100 (sm_60) Kaggle hands out.
# Must run before any torch import, since a live kernel keeps whichever version it loaded.
set -e
nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | sed 's/^/card: /'
pip install -q --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
python -c "import torch; print('torch', torch.__version__)"


In [ ]:
import os

# 2026-08-08: five sessions in a row drew a P100, including one pushed through the API with
# machine_shape=gpuT4x2 and --accelerator gpuT4x2. The accelerator field is a request the
# scheduler may ignore, so waiting for a T4 is not a plan. The remaining weekly GPU quota is
# spent on the slower card instead, and the card actually used is printed below so the run
# states its own hardware.
os.environ['MONARCH_ALLOW_PRE_VOLTA'] = '1'

import torch

assert torch.cuda.is_available(), 'No GPU. Settings -> Accelerator -> GPU, then restart.'

# A card can be visible and still unusable: torch only runs kernels for the architectures its
# build targets, and a mismatch surfaces deep in the model rather than here.
major, minor = torch.cuda.get_device_capability(0)
arch = f'sm_{major}{minor}'
supported = torch.cuda.get_arch_list()
assert arch in supported, (
    f'{torch.cuda.get_device_name(0)} is {arch}, which torch {torch.__version__} '
    f'does not target. Built for: {supported}'
)

# Kaggle hands out a P100 whenever two T4s are not free, whatever the metadata asked for.
# Pre-Volta costs the fp16 path in ctranslate2 and roughly doubles the per-item time, so a
# session that lands on one is worth abandoning in seconds rather than at hour three.
MIN_CAPABILITY = 7
if major < MIN_CAPABILITY and os.environ.get('MONARCH_ALLOW_PRE_VOLTA') != '1':
    # Printed before raising: papermill reports SystemExit as "An exception has occurred"
    # and swallows its message, which makes a deliberate stop look unexplained.
    print(f'STOP: {torch.cuda.get_device_name(0)} is {arch}, below sm_70.', flush=True)
    print('  Kaggle allocates a P100 whenever two T4s are not free.', flush=True)
    print('  Set Accelerator to GPU T4 x2 and start the session again,', flush=True)
    print('  or set MONARCH_ALLOW_PRE_VOLTA=1 to run here at about half the speed.', flush=True)
    raise SystemExit('pre-Volta GPU refused')

if major < MIN_CAPABILITY:
    print(f'PRE-VOLTA ACCEPTED: {torch.cuda.get_device_name(0)} is {arch}. '
          'Expect about 190 s/item, not 110.', flush=True)

print(torch.__version__, torch.cuda.get_device_name(0), arch, 'OK')
print('device count:', torch.cuda.device_count())


In [ ]:
%%bash
# Code lives in /kaggle/temp, not /kaggle/working. Everything under working becomes kernel
# output, so cloning there put the whole repository plus tribev2 in every download.
# The branch matters: --carry-cols and the corpus-builder fixes live only there.
set -e
REPO_URL=https://github.com/brn-mwai/monarch.git
BRANCH=thesis/amendment-and-analysis-layer
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf monarch tribev2
git clone -q --branch $BRANCH $REPO_URL monarch
git clone -q https://github.com/brn-mwai/tribev2.git
apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y ffmpeg > /dev/null
cd monarch && git log --oneline -1
# Fail here rather than after eight GPU-hours: an older checkout drops every carried column.
grep -q 'carry-cols' services/inference/scripts/batch_naa.py \
  && echo 'OK: batch_naa.py supports --carry-cols' \
  || { echo 'STOP: this checkout predates --carry-cols. Push the thesis branch first.'; exit 1; }


In [ ]:
%%bash
set -e
# Pin the three torch packages so resolving tribev2's dependencies cannot pull a different
# build in behind us; everything else is free to resolve.
mkdir -p /kaggle/temp
printf 'torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
' > /kaggle/temp/constraints.txt

# Oldest published version satisfying neuralset's exca>=0.5.20 floor, so the API it
# calls exists without jumping to a release it was never tested against.
pip install -q exca==0.5.21

# WITH dependencies this time. Installing tribev2 --no-deps left neuralset, neuraltrain,
# einops, moviepy, soundfile and julius missing, and the smoke test died on the first import.
pip install -q -c /kaggle/temp/constraints.txt /kaggle/temp/tribev2

# whisperx wants torch ~=2.8 and would drag the pinned build out; it only needs to run as a
# subprocess for word timings, so it goes in without its dependency graph.
# whisperx 3.8.x demands torch~=2.8 while tribev2 demands <2.7, so they cannot coexist and
# --no-deps was the old way round it. That left pyannote uninstalled, and whisperx imports it
# at module load, so the CLI died the moment tribev2 shelled out to it for word timings.
# 3.4.2 is the newest release whose floor (torch>=2.5.1, pyannote-audio>=3.3.2,
# ctranslate2<4.5.0) the pinned build satisfies, so it installs WITH its dependencies.
pip install -q -c /kaggle/temp/constraints.txt "whisperx==3.4.2"
pip install -q -c /kaggle/temp/constraints.txt nltk nibabel ujson mne torchmetrics

# whisperx pins ctranslate2<4.5.0, which links cuDNN 8, while torch 2.5.1+cu121 ships cuDNN 9.
# The mismatch does not surface at import: it appears when the transcriber loads a model on
# the GPU, as "Could not load library libcudnn_ops_infer.so.8". 4.5.0 is the release that
# moved to cuDNN 9, and faster-whisper accepts anything in >=4.0,<5, so the upper pin is
# simply stale for this environment. --no-deps keeps the resolution from touching torch.
pip install -q --no-deps "ctranslate2==4.5.0"

python -m spacy download en_core_web_lg -q
python -c "import neuralset, neuraltrain, tribev2; print('tribev2 stack imports OK')"
# tribev2 runs whisperx as a subprocess, so what matters is that the CLI imports, not that the
# package is present. This is the check that would have caught the pyannote failure at install
# time instead of part way into the first item's inference.
whisperx --help > /dev/null && echo "whisperx CLI imports OK"

In [ ]:
import importlib.metadata as md

# neuralset 0.0.2 declares exca>=0.5.20 and calls exca.steps.base.NoValue, which does not
# exist in 0.5.17. The previous notebook pinned 0.5.17 and rewrote neuralset's version guard
# to accept it, which silenced the check without supplying the API and failed at import.
# The versions actually resolved are printed here so a run states its own environment.
for pkg in ("exca", "neuralset", "neuraltrain", "tribev2", "torch", "numpy", "transformers"):
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} NOT INSTALLED")

import exca.steps.base as base

assert hasattr(base, "NoValue"), (
    f"exca {md.version('exca')} has no steps.base.NoValue; neuralset needs >=0.5.20"
)
print("exca API check OK")


In [ ]:
import sys

sys.path.insert(0, '/kaggle/temp/monarch/services/inference')

from scripts.kaggle_bootstrap import apply_session_environment

SESSION = apply_session_environment()


In [ ]:
%%bash
# tribev2 runs whisperx as a subprocess, so the CLI has to start, reach the GPU and run a
# convolution. Loading a model does none of that, which is why an earlier import-only check
# passed while inference failed. tiny weights keep this to about a minute against the eight
# hours it protects; the precision comes from the session environment, so this probes the
# same configuration the scan will use.
set -e
cd /kaggle/temp
python3 - <<'PROBE'
import math
import struct
import wave

with wave.open("/kaggle/temp/probe.wav", "w") as handle:
    handle.setnchannels(1)
    handle.setsampwidth(2)
    handle.setframerate(16000)
    handle.writeframes(b"".join(struct.pack("<h", int(3000 * math.sin(i / 8))) for i in range(16000)))
print("probe.wav written")
PROBE

echo "compute_type: $MONARCH_WHISPER_COMPUTE"
whisperx /kaggle/temp/probe.wav --model tiny --language en --device cuda \
  --compute_type "$MONARCH_WHISPER_COMPUTE" \
  --output_dir /kaggle/temp/probe_out --output_format json
echo "whisperx GPU transcription OK"


In [ ]:
%%bash
cd /kaggle/temp/monarch/services/inference
PYTHONPATH=/kaggle/temp/tribev2 python scripts/smoke_test.py

Expect `Model loaded on device: cuda:0`, `Predictions shape: (T, 20484)`, `Smoke test PASSED`.
If this fails, stop. Every later cell burns GPU hours on a broken cascade.

In [ ]:
import csv
import glob
import shutil
from collections import Counter

matches = sorted(glob.glob('/kaggle/input/**/corpus.csv', recursive=True))
assert matches, 'corpus.csv not found. Add data -> your corpus dataset.'
corpus_src = matches[0]
shutil.copy(corpus_src, '/kaggle/working/corpus.csv')

with open('/kaggle/working/corpus.csv', newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
counts = Counter(r['category'] for r in rows)
print(corpus_src, '->', len(rows), 'rows')
for category, n in sorted(counts.items()):
    print(f'  {category:24s} {n}')
assert len(rows) == 400, f'expected 400 rows, got {len(rows)}'
assert len(counts) == 4 and set(counts.values()) == {100}, f'category imbalance: {counts}'

# Seed the output from a previous partial run so batch_naa resumes instead of rescanning.
partials = sorted(glob.glob('/kaggle/input/**/corpus_naa.csv', recursive=True))
if partials:
    shutil.copy(partials[0], '/kaggle/working/corpus_naa.csv')
    with open('/kaggle/working/corpus_naa.csv', newline='', encoding='utf-8') as handle:
        done = sum(1 for _ in csv.DictReader(handle))
    print(f'resuming from {partials[0]}: {done} items already scanned')
else:
    print('no partial found; starting a fresh scan')

In [ ]:
%%bash
# The 400-item corpus is complete (v36). This run captures what that scan threw away.
#
# batch_naa.py reduced each item's (20484,) prediction to two ROI means and discarded the
# vector, so no per-vertex map exists for any scanned item. The surface view can therefore
# only paint two flat regions. --save-vectors keeps the map.
#
# A fresh --out so nothing is skipped by resume, and --limit 12 because 12 items at the
# measured 64 to 78 s is about 15 minutes and is enough to show real structure. The full
# corpus would cost another 7 to 9 hours for a picture, not for a number.
cd /kaggle/temp/monarch/services/inference
PYTHONPATH=/kaggle/temp/tribev2 python scripts/batch_naa.py \
  --csv /kaggle/working/corpus.csv \
  --text-col text \
  --outcome-col category \
  --limit 12 \
  --save-vectors /kaggle/working/vectors \
  --carry-cols id,manipulative,credibility,partisan_intensity,source_dataset,word_count \
  --out /kaggle/working/corpus_vectors_naa.csv

In [ ]:
import csv
import statistics
from collections import Counter, defaultdict

with open('/kaggle/working/corpus_naa.csv', newline='', encoding='utf-8') as handle:
    scanned = list(csv.DictReader(handle))

print(f'rows scanned: {len(scanned)} / 400')

# The ratio form of NAA is undefined whenever either network mean sits below baseline. How
# often that happens is a finding in itself and goes in the methods, so it is counted here.
defined = [r for r in scanned if r.get('naa')]
print(f'NAA (ratio) defined: {len(defined)}  undefined: {len(scanned) - len(defined)}')

by_category = defaultdict(list)
for r in scanned:
    if r.get('naa_signed'):
        by_category[r['category']].append(float(r['naa_signed']))

print('\nsigned NAA (A_aff - A_del) per category, descriptive only:')
for category, values in sorted(by_category.items()):
    if not values:
        continue
    spread = statistics.stdev(values) if len(values) > 1 else float('nan')
    print(f'  {category:24s} n={len(values):3d}  median={statistics.median(values):+.4f}  sd={spread:.4f}')

print('\nclassification counts:', dict(Counter(r.get('classification', '') for r in scanned)))
print('\nNo effect is claimed here. Run scripts/analyze_corpus.py on this CSV for RQ I / RQ II.')

In [ ]:
%%bash
# Runs after the scan on purpose: it is evidence for the methods section, not a
# prerequisite, and an earlier ordering let it block the measurement.
# Chapter 4 has to state the model's depth and how it treats subject identity. Record both
# from the loaded artifact, not from the run-folder name, and keep the JSON with the results.
cd /kaggle/temp/monarch/services/inference
PYTHONPATH=/kaggle/temp/tribev2 python scripts/verify_tribe_checkpoint.py \
  --load-model --out /kaggle/working/tribe_facts.json

## When this finishes

Download from the right panel (Output): `corpus_naa.csv` and `tribe_facts.json`.

If the session died partway, `corpus_naa.csv` still holds every item scanned so far, flushed
per row. Publish it as a dataset, attach it to the next run, and cell 8 picks it up.

**Do not quote an `alpha_hat` from this run.** Calibration is a separate CPU step
(`scripts/calibrate_alpha.py`), and the two prior runs returned an interval straddling zero.
If it straddles again, that null is the result and gets reported with its power statement.